In [ ]:
# ================================
# Transforms raw invoice data into clean,
#item-level dataset with extracted SKU,
#deduplicated phone/email, and formatted datetime
# ================================
import pandas as pd
import json
import re
from google.colab import files

# ================================
# UPLOAD FILE
# ================================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

if file_name.endswith('.csv'):
    df = pd.read_csv(file_name)
else:
    df = pd.read_excel(file_name)

print("File Loaded:", df.shape)

# ================================
# HELPER FUNCTIONS
# ================================
def clean_phone(val):
    if pd.isna(val):
        return None
    val = str(val)
    val = re.sub(r'\.0$', '', val)
    val = re.sub(r'\D', '', val)
    return val[-10:] if len(val) >= 10 else None

def extract_email(val):
    if pd.isna(val):
        return None
    val = str(val)
    return val.lower() if "@" in val else None

def safe_json(x):
    try:
        return json.loads(x)
    except:
        return []

# ================================
# EXPLODE ITEMS
# ================================
df['items_parsed'] = df['items'].apply(safe_json)
df = df.explode('items_parsed').reset_index(drop=True)

# ================================
# ITEM EXTRACTION
# ================================
df['Invoice Brand'] = df['items_parsed'].apply(lambda x: x.get('Items|Invoice Brand') if isinstance(x, dict) else None)
df['Model Number'] = df['items_parsed'].apply(lambda x: x.get('Items|Model Number') if isinstance(x, dict) else None)
df['MRP'] = df['items_parsed'].apply(lambda x: x.get('Items|MRP') if isinstance(x, dict) else None)
df['Billing Value'] = df['items_parsed'].apply(lambda x: x.get('Items|Billing Value') if isinstance(x, dict) else None)
df['Billing Discount Percentage'] = df['items_parsed'].apply(lambda x: x.get('Items|Billing Discount Percentage') if isinstance(x, dict) else None)

# ================================
# EXTRACT ALL CONTACTS
# ================================
def extract_all_contacts(row):
    phones = []
    emails = []

    # profile.all_identities
    try:
        ids = json.loads(row['profile.all_identities'])
    except:
        ids = []

    for i in ids:
        if "@" in str(i):
            emails.append(str(i).lower())
        else:
            phone = clean_phone(i)
            if phone:
                phones.append(phone)

    # profile.phone
    if pd.notna(row.get('profile.phone')):
        phone = clean_phone(row['profile.phone'])
        if phone:
            phones.append(phone)

    # profile.identity
    if pd.notna(row.get('profile.identity')):
        phone = clean_phone(row['profile.identity'])
        if phone:
            phones.append(phone)

    # profile.email
    if pd.notna(row.get('profile.email')):
        email = extract_email(row['profile.email'])
        if email:
            emails.append(email)

    return list(set(phones)), list(set(emails))

df[['phones', 'emails']] = df.apply(lambda row: pd.Series(extract_all_contacts(row)), axis=1)

# ================================
# NAME EXTRACTION
# ================================
def extract_names(row):
    names = []

    if pd.notna(row.get('profile.name')):
        names.append(str(row['profile.name']).strip())

    if pd.notna(row.get('profile.profileData.customername')):
        names.append(str(row['profile.profileData.customername']).strip())

    # remove duplicates (case insensitive)
    names = list(set([n.lower() for n in names if n]))

    return names

df['names'] = df.apply(extract_names, axis=1)

# ================================
# DYNAMIC COLUMNS
# ================================
max_phones = df['phones'].apply(len).max()
max_emails = df['emails'].apply(len).max()
max_names = df['names'].apply(len).max()

for i in range(max_phones):
    df[f'phone{i+1}'] = df['phones'].apply(lambda x: x[i] if i < len(x) else None)

for i in range(max_emails):
    df[f'email{i+1}'] = df['emails'].apply(lambda x: x[i] if i < len(x) else None)

for i in range(max_names):
    df[f'name{i+1}'] = df['names'].apply(lambda x: x[i] if i < len(x) else None)

# ================================
# PRIMARY NAME
# ================================
df['primary_name'] = df['name1']

# ================================
# DATE CONVERSION
# ================================
df['invoice_datetime'] = pd.to_datetime(df['event_props.Invoice Date'], unit='s', errors='coerce')
df['final_datetime'] = pd.to_datetime(df['ts'], format='%Y%m%d%H%M%S', errors='coerce')

# ================================
# FINAL OUTPUT
# ================================
base_cols = [
    'event_props.CUID Number',
    'event_props.Billing Store Zone',
    'event_props.Invoice Number',
    'event_props.Invoice Type',
    'event_props.Store City',
    'event_props.Store Code',
    'event_props.Tender Name',
    'Invoice Brand',
    'Model Number',
    'MRP',
    'Billing Value',
    'Billing Discount Percentage',
    'primary_name',
    'invoice_datetime',
    'final_datetime'
]

phone_cols = [col for col in df.columns if col.startswith('phone')]
email_cols = [col for col in df.columns if col.startswith('email')]
name_cols = [col for col in df.columns if col.startswith('name')]

final_df = df[base_cols + name_cols + phone_cols + email_cols]

# ================================
# EXPORT
# ================================
final_df.to_csv("final_cleaned_data.csv", index=False)
files.download("final_cleaned_data.csv")

print("✅ Done: Full cleaned dataset with dynamic contacts + primary name ready")